# Comparing Text Representations: Counts and Embeddings — companion notebook

Chapter 4 of *Applied Natural Language Processing — From Tokens to Agents*.
Read the chapter first: **[Comparing Text Representations: Counts and Embeddings](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html)**.

The chapter puts the counting methods of chapter 2 and the embedding methods of chapter 3
on one task, with one split, one classifier and one metric, and then asks what the
resulting table does and does not show. This notebook runs that comparison in the
chapter's order:

1. **The data and the split**: 235,165 Turkish product reviews, the checks before
   de-duplication, and a fingerprint of the rows in each split.
2. **The class error the first score hides**: what a model that never predicts "negative"
   scores.
3. **Tokenising Turkish**: the dotted and dotless i, and what `str.lower()` does to them.
4. **Round 1**: bag of words, TF-IDF, word bigrams and character n-grams.
5. **Round 2**: averaged word2vec, averaged fastText and doc2vec, trained on the same
   40,000 reviews.
6. **Intervals**: a paired bootstrap for every adjacent pair of rows.
7. **Out-of-vocabulary words**, error examples and the cosine between a review and its negation.
8. **The crossover**: the same comparison with 100 to 40,000 labels.
9. **Cost, a hybrid that does not help, and word order.**

**How to read the numbers.** Everything printed below is what *this run* produced. Where
the chapter prints the same quantity, the cell prints the book's value beside it and says
whether the two match at the book's printed precision. The sparse rows (counts, TF-IDF)
reproduce exactly. The dense rows (word2vec, fastText, doc2vec) are the chapter's caveat:
its round 2 section says they cannot be reproduced to the last digit, and gives the
values of a second run of its own. For those rows the cells report the distance from the
book's value instead of a match; section 6 shows which of the three actually moves from
run to run. Timings depend on your machine; read them as orders of magnitude.

**Runtime and downloads.** About 7 minutes on a laptop CPU (Apple M-series) for a full run.
With `ANLP_SKIP_HEAVY=1`, about 3 minutes. The only download is the dataset (24 MB), cached
under `data/cache/` next to this notebook.

**`ANLP_SKIP_HEAVY=1`** (CI sets it) shortens three things instead of skipping them: the
bootstraps draw 1,000 resamples instead of 10,000, the seed-spread cell trains one seed
instead of five, and the hybrid's validation grid is reduced to the one point the
chapter's search chose. Those cells say so and do not compare themselves with the book.

## 1. Setup

The first cell records the interpreter and package versions this run used. The chapter's
status note names `scikit-learn` 1.9 and `gensim` 4.4. If your numbers differ from the
ones saved in this notebook, compare this cell first.

In [1]:
import collections
import hashlib
import os
import platform
import re
import time
import unicodedata
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import scipy.sparse as sp
from scipy.stats import binomtest
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

SEED = 42
SKIP_HEAVY = os.environ.get("ANLP_SKIP_HEAVY") == "1"
N_BOOT = 1_000 if SKIP_HEAVY else 10_000

print(f"Python        {platform.python_version()}  ({platform.system()} {platform.machine()})")
for pkg in ["numpy", "pandas", "scipy", "scikit-learn", "gensim", "pyarrow", "requests"]:
    print(f"{pkg:<13} {version(pkg)}")
print("\nANLP_SKIP_HEAVY", "= 1: some cells run a shortened version" if SKIP_HEAVY
      else "is not set: every cell runs in full")

Python        3.12.13  (Darwin arm64)
numpy         2.5.2
pandas        3.0.5
scipy         1.18.1
scikit-learn  1.9.0
gensim        4.4.0
pyarrow       25.0.1
requests      2.34.2

ANLP_SKIP_HEAVY is not set: every cell runs in full


Three helpers used throughout.

- `vs_book` prints this run's number beside the book's and compares them at the book's
  printed precision.
- `vs_book_near` is for the dense rows. It prints the difference from the book's value
  and says whether it is inside a stated tolerance.
- `scores` returns accuracy, macro-F1 and the F1 of the negative class, in percent. The
  macro-F1 call is the chapter's: `f1_score(..., average="macro")`.

In [2]:
def vs_book(label, ours, book, fmt="{}", note=None):
    """This run's value beside the book's, compared at the book's printed precision."""
    ours_s, book_s = fmt.format(ours), fmt.format(book)
    if note:
        verdict = f"not comparable: {note}"
    else:
        verdict = "same as the book" if ours_s == book_s else "DIFFERS from the book"
    print(f"{label:<38} this run {ours_s:>10}   book {book_s:>10}   {verdict}")


def vs_book_near(label, ours, book, tol, fmt="{:.2f}", note=None):
    """For rows that are not bit-reproducible: the distance from the book's value."""
    if note:
        verdict = f"not comparable: {note}"
    else:
        verdict = (f"{ours - book:+.2f} from the book, "
                   + ("inside" if abs(ours - book) <= tol else "OUTSIDE") + f" ±{tol}")
    print(f"{label:<38} this run {fmt.format(ours):>10}   book {fmt.format(book):>10}   {verdict}")


def scores(y_true, y_pred):
    """Accuracy, macro-F1 and negative-class F1, in percent."""
    return (100 * accuracy_score(y_true, y_pred),
            100 * f1_score(y_true, y_pred, average="macro", zero_division=0),
            100 * f1_score(y_true, y_pred, pos_label=0, zero_division=0))

## 2. The problem: the data and the split

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#the-problem)

The corpus is `fthbrmnby/turkish_product_reviews` on Hugging Face: Turkish product reviews
labelled positive or negative, compiled by Fatih Barmanbay and distributed under
**CC BY-SA 4.0**. The chapter notes what the dataset card does not say: which site the
reviews came from, when they were collected, and how the positive/negative label was
derived.

The loading code is the chapter's. The only change is a local cache, so that re-running
the notebook does not download the file again. The chapter runs three checks *before*
de-duplicating, because `drop_duplicates` silently decides what happens to a text that
carries both labels: it keeps the first row.

**What to look at:** 21 texts are empty after stripping, and 77 texts appear with both
labels.

In [3]:
URL = ("https://huggingface.co/datasets/fthbrmnby/turkish_product_reviews/"
       "resolve/main/data/train-00000-of-00001.parquet")
CACHE = Path("data/cache/turkish_product_reviews.parquet")

if not CACHE.exists():
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    response = requests.get(URL, timeout=300)
    response.raise_for_status()
    CACHE.write_bytes(response.content)
print(f"parquet file  {CACHE.stat().st_size:,} bytes, "
      f"sha256 {hashlib.sha256(CACHE.read_bytes()).hexdigest()[:12]}\n")

# the chapter's loading code
df = pd.read_parquet(CACHE).rename(
    columns={"sentence": "text", "sentiment": "label"})
n_raw = len(df)
n_missing = int(df.isna().any(axis=1).sum())
df["text"] = df["text"].astype(str).str.strip()
labels_per_text = df.groupby("text")["label"].nunique()
n_empty = int(df["text"].eq("").sum())
n_both = int(labels_per_text.gt(1).sum())
df = df.drop_duplicates(subset="text")

print("### checks before de-duplication")
vs_book("missing text or label", n_missing, 0, "{:,}")
vs_book("empty texts", n_empty, 21, "{:,}")
vs_book("texts with both labels", n_both, 77, "{:,}")
print("\n### the raw data")
vs_book("rows as downloaded", n_raw, 235_165, "{:,}")
vs_book("after de-duplication", len(df), 233_508, "{:,}")
vs_book("exact duplicates removed", n_raw - len(df), 1_657, "{:,}")
vs_book("label 1 (positive)", int((df["label"] == 1).sum()), 218_747, "{:,}")
vs_book("label 0 (negative)", int((df["label"] == 0).sum()), 14_761, "{:,}")
vs_book("positive share, %", 100 * (df["label"] == 1).mean(), 93.7, "{:.1f}")

parquet file  24,354,762 bytes, sha256 a5ea59cd4f78

### checks before de-duplication
missing text or label                  this run          0   book          0   same as the book
empty texts                            this run         21   book         21   same as the book
texts with both labels                 this run         77   book         77   same as the book

### the raw data
rows as downloaded                     this run    235,165   book    235,165   same as the book
after de-duplication                   this run    233,508   book    233,508   same as the book
exact duplicates removed               this run      1,657   book      1,657   same as the book
label 1 (positive)                     this run    218,747   book    218,747   same as the book
label 0 (negative)                     this run     14,761   book     14,761   same as the book
positive share, %                      this run       93.7   book       93.7   same as the book


### The split, with a fingerprint

A stratified sample of 50,000 reviews, then a stratified 80/20 split, both with seed 42.
"Stratified, seed 42" names a family of procedures, so the chapter prints the two calls
and a fingerprint of the row indices in each part. If your fingerprints match, you have
the same 40,000 training and 10,000 test reviews, row for row.

In [4]:
# the chapter's split
sub, _ = train_test_split(df, train_size=50_000,
                          stratify=df["label"], random_state=42)
train, test = train_test_split(sub, test_size=0.2,
                               stratify=sub["label"], random_state=42)

def fingerprint(frame):
    rows = ",".join(map(str, sorted(frame.index)))
    return hashlib.sha256(rows.encode()).hexdigest()[:12]

print("### the split (stratified, seed 42)")
for name, part in [("train", train), ("test", test)]:
    pos = int(part["label"].sum())
    print(f"{name:<6} {len(part):>7,}   {pos:>7,} positive   "
          f"{len(part) - pos:>6,} negative   rows {fingerprint(part)}")

y_train, y_test = train["label"].to_numpy(), test["label"].to_numpy()
print()
vs_book("train fingerprint", fingerprint(train), "8205c2be5ebf")
vs_book("test fingerprint", fingerprint(test), "634cdb3bb002")
vs_book("negatives in train", int((y_train == 0).sum()), 2_529, "{:,}")
vs_book("negatives in test", int((y_test == 0).sum()), 632, "{:,}")

### the split (stratified, seed 42)
train   40,000    37,471 positive    2,529 negative   rows 8205c2be5ebf
test    10,000     9,368 positive      632 negative   rows 634cdb3bb002

train fingerprint                      this run 8205c2be5ebf   book 8205c2be5ebf   same as the book
test fingerprint                       this run 634cdb3bb002   book 634cdb3bb002   same as the book
negatives in train                     this run      2,529   book      2,529   same as the book
negatives in test                      this run        632   book        632   same as the book


## 3. The class error the first score hides

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#the-trap-in-the-first-number-you-will-compute)

Predict "positive" for every review and never look at the text. The chapter prints this
model's confusion matrix in full because its accuracy looks like a result.

**What to look at:** accuracy equals the share of positive reviews in the test set, and
the F1 of the negative class is zero: the model finds none of the 632 negative reviews.
Macro-F1 is the unweighted mean of the two class F1 scores, so it halves.

In [5]:
majority_pred = np.ones_like(y_test)
cm = confusion_matrix(y_test, majority_pred, labels=[0, 1])
print("### majority baseline, in full")
print("                 predicted neg   predicted pos   total")
for i, name in enumerate(["actual neg", "actual pos"]):
    print(f"   {name}   {cm[i, 0]:>8,}   {cm[i, 1]:>13,}   {cm[i].sum():>10,}")
f1_neg = f1_score(y_test, majority_pred, pos_label=0, zero_division=0)
f1_pos = f1_score(y_test, majority_pred, pos_label=1)
print()
majority_acc, majority_macro, _ = scores(y_test, majority_pred)
vs_book("accuracy, %", majority_acc, 93.68, "{:.2f}")
vs_book("F1 (neg)", 100 * f1_neg, 0.00, "{:.2f}")
vs_book("F1 (pos)", 100 * f1_pos, 96.74, "{:.2f}")
vs_book("macro-F1", majority_macro, 48.37, "{:.2f}")

### majority baseline, in full
                 predicted neg   predicted pos   total
   actual neg          0             632          632
   actual pos          0           9,368        9,368

accuracy, %                            this run      93.68   book      93.68   same as the book
F1 (neg)                               this run       0.00   book       0.00   same as the book
F1 (pos)                               this run      96.74   book      96.74   same as the book
macro-F1                               this run      48.37   book      48.37   same as the book


**Reading it.** The model never predicts negative, so the precision of the negative class
is 0/0. The chapter reports it as *undefined* and gives F1 through the count form
$2\,TP / (2\,TP + FP + FN) = 0 / 632$, which is defined. `scikit-learn` needs
`zero_division=0` only to silence its warning about the undefined precision.

Every row of every table below has to be read against **48.37 macro-F1**, not against zero.

## 4. Tokenising Turkish

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#tokenising-turkish-and-the-trap-inside-that)

Turkish has four letters where English has two: dotted `İ`/`i` and dotless `I`/`ı`.
Python's `str.lower()` follows the default Unicode rule, which is not the Turkish one. It
maps `I` to `i` (merging two different letters) and maps `İ` to `i` followed by a
combining dot, U+0307.

**What to look at:** the length of `"İYİ".lower()`. Three letters go in and five code
points come out.

In [6]:
TR_LOWER = str.maketrans({"I": "ı", "İ": "i"})
def tr_lower(s):
    return s.translate(TR_LOWER).lower()

print("### the same string, two lowercasing rules")
print(f"{'':<12}{'str.lower()':<26}{'tr_lower()'}")
for s in ["IYI", "İYİ"]:
    a, b = s.lower(), tr_lower(s)
    print(f'"{s}"       {a!r:<8} {len(a)} code points     {b!r:<8} {len(b)} code points')
print("\ncode points of 'İYİ'.lower():",
      " ".join(f"U+{ord(c):04X}" for c in "İYİ".lower()))

### the same string, two lowercasing rules
            str.lower()               tr_lower()
"IYI"       'iyi'    3 code points     'ıyı'    3 code points
"İYİ"       'i̇yi̇'  5 code points     'iyi'    3 code points

code points of 'İYİ'.lower(): U+0069 U+0307 U+0079 U+0069 U+0307


The chapter's tokenizer: lowercase with the Turkish rule, then keep runs of Turkish
lowercase letters. Digits, punctuation and emoji are dropped, and so is the combining dot,
which is why a word that contains it is split in two.

The printing below is ours; the statistics are the chapter's table for the 40,000
training reviews.

In [7]:
TOKEN_RE = re.compile(r"[a-zçğıöşü]+")
def tokenize(s):
    return TOKEN_RE.findall(tr_lower(s))

train_tok = [tokenize(s) for s in train["text"]]
test_tok = [tokenize(s) for s in test["text"]]
counts = collections.Counter(w for doc in train_tok for w in doc)
lengths = np.array([len(doc) for doc in train_tok])
hapax = sum(1 for c in counts.values() if c == 1)

print("### the training corpus, 40,000 reviews")
vs_book("tokens", sum(counts.values()), 903_190, "{:,}")
vs_book("distinct word forms", len(counts), 74_317, "{:,}")
vs_book("hapax (seen exactly once)", hapax, 43_579, "{:,}")
vs_book("hapax share of forms, %", 100 * hapax / len(counts), 58.6, "{:.1f}")
vs_book("mean review length, tokens", lengths.mean(), 22.6, "{:.1f}")
vs_book("median", int(np.median(lengths)), 19)
vs_book("95th percentile", int(np.percentile(lengths, 95)), 56)
vs_book("longest", int(lengths.max()), 296)

### the training corpus, 40,000 reviews
tokens                                 this run    903,190   book    903,190   same as the book
distinct word forms                    this run     74,317   book     74,317   same as the book
hapax (seen exactly once)              this run     43,579   book     43,579   same as the book
hapax share of forms, %                this run       58.6   book       58.6   same as the book
mean review length, tokens             this run       22.6   book       22.6   same as the book
median                                 this run         19   book         19   same as the book
95th percentile                        this run         56   book         56   same as the book
longest                                this run        296   book        296   same as the book


More than half of all word forms are seen exactly once. One reason is morphology: one
Turkish stem appears in hundreds of surface forms. The chapter counts the distinct forms
that share a five-character prefix.

**What to look at:** `kulla` (the stem of *kullanmak*, to use) alone accounts for 895
columns of a word-level vocabulary.

In [8]:
by_prefix = collections.defaultdict(list)
for form, count in counts.most_common():          # most frequent form first
    if len(form) >= 5:
        by_prefix[form[:5]].append(form)

print("### distinct word forms sharing a five-character prefix")
for prefix, book in [("kulla", 895), ("bekle", 285), ("çalış", 247), ("çıkar", 209), ("düşün", 195)]:
    forms = by_prefix[prefix]
    print(f"{prefix}   {', '.join(forms[:5])}, ...")
    vs_book(f"  forms starting with {prefix}", len(forms), book)

### distinct word forms sharing a five-character prefix
kulla   kullanışlı, kullanıyorum, kullandım, kullanımı, kullanım, ...
  forms starting with kulla            this run        895   book        895   same as the book
bekle   beklediğimden, beklediğim, beklemiyordum, beklemeyin, beklentimin, ...
  forms starting with bekle            this run        285   book        285   same as the book
çalış   çalışıyor, çalışma, çalışan, çalışması, çalışırken, ...
  forms starting with çalış            this run        247   book        247   same as the book
çıkar   çıkarıyor, çıkar, çıkartıyor, çıkarmıyor, çıkardı, ...
  forms starting with çıkar            this run        209   book        209   same as the book
düşün   düşünüyorum, düşünmeden, düşünenlere, düşünen, düşünmeyin, ...
  forms starting with düşün            this run        195   book        195   same as the book


## 5. Round 1: the chapter 2 methods

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#round-1-the-chapter-2-methods)

Four ways of counting, one classifier. The vectorisers are the chapter's, including the
two arguments that keep `scikit-learn` from calling `str.lower()` itself. Every
vectoriser is fitted on the training reviews only and then applied to the test reviews.

The classifier is the same for every row of the chapter: logistic regression with
`class_weight="balanced"`, which weights the 2,529 negative training reviews as heavily
as the 37,471 positive ones.

The `fit` column is the time to build the training matrix and fit the classifier, on this
machine.

**What to look at:** accuracy and macro-F1 rank the rows differently. Three of the four
models have *lower* accuracy than the majority baseline and far higher macro-F1.

In [9]:
PAT = r"[a-zçğıöşü]+"
# lowercase=False + preprocessor=tr_lower: otherwise scikit-learn calls
# Python's str.lower() and commits the bug we warned about above
LOC = dict(preprocessor=tr_lower, lowercase=False)

def make_schemes():
    return {
      "bag of words":    CountVectorizer(min_df=2, token_pattern=PAT, **LOC),
      "tf-idf":          TfidfVectorizer(min_df=2, sublinear_tf=True, token_pattern=PAT, **LOC),
      "tf-idf 1-2 gram": TfidfVectorizer(min_df=2, sublinear_tf=True,
                                         ngram_range=(1, 2), token_pattern=PAT, **LOC),
        # a separate feature-extraction path: token_pattern is not used, and min_df differs
      "tf-idf char_wb 3-5": TfidfVectorizer(min_df=3, sublinear_tf=True,
                                            analyzer="char_wb", ngram_range=(3, 5), **LOC),
    }

def new_clf(C=1.0, max_iter=2000):
    return LogisticRegression(C=C, max_iter=max_iter, class_weight="balanced", random_state=42)

BOOK_ROUND1 = {"bag of words": (92.14, 73.51, 51.30, 30_195), "tf-idf": (90.07, 71.95, 49.41, 30_195),
               "tf-idf 1-2 gram": (92.57, 76.21, 56.47, 101_659), "tf-idf char_wb 3-5": (88.63, 70.39, 47.14, 94_939)}

schemes = make_schemes()
X_train, X_test, pred, cost = {}, {}, {}, {}
print("### round 1: counting")
print(f"{'':<22}accuracy   macro-F1   F1(neg)      dims      fit")
print(f"{'majority baseline':<22}{majority_acc:8.2f} {majority_macro:10.2f} {0:9.2f}         —        —")
for name, vec in schemes.items():
    start = time.perf_counter()
    X_train[name] = vec.fit_transform(train["text"])
    t_build = time.perf_counter() - start
    start = time.perf_counter()
    X_test[name] = vec.transform(test["text"])
    t_test = time.perf_counter() - start
    start = time.perf_counter()
    model = new_clf().fit(X_train[name], y_train)
    t_fit = time.perf_counter() - start
    pred[name] = model.predict(X_test[name])
    cost[name] = (t_build, t_test, X_train[name])
    acc, macro, f1n = scores(y_test, pred[name])
    print(f"{name:<22}{acc:8.2f} {macro:10.2f} {f1n:9.2f} {X_train[name].shape[1]:>9,} {t_build + t_fit:7.1f} s")

print()
for name, (b_acc, b_macro, b_f1n, b_dims) in BOOK_ROUND1.items():
    acc, macro, f1n = scores(y_test, pred[name])
    vs_book(f"{name}: accuracy", acc, b_acc, "{:.2f}")
    vs_book(f"{name}: macro-F1", macro, b_macro, "{:.2f}")
    vs_book(f"{name}: F1(neg)", f1n, b_f1n, "{:.2f}")
    vs_book(f"{name}: dims", X_train[name].shape[1], b_dims, "{:,}")

### round 1: counting
                      accuracy   macro-F1   F1(neg)      dims      fit
majority baseline        93.68      48.37      0.00         —        —


bag of words             92.14      73.51     51.30    30,195     0.7 s


tf-idf                   90.07      71.95     49.41    30,195     0.5 s


tf-idf 1-2 gram          92.57      76.21     56.47   101,659     1.0 s


tf-idf char_wb 3-5       88.63      70.39     47.14    94,939     3.2 s

bag of words: accuracy                 this run      92.14   book      92.14   same as the book
bag of words: macro-F1                 this run      73.51   book      73.51   same as the book
bag of words: F1(neg)                  this run      51.30   book      51.30   same as the book
bag of words: dims                     this run     30,195   book     30,195   same as the book
tf-idf: accuracy                       this run      90.07   book      90.07   same as the book
tf-idf: macro-F1                       this run      71.95   book      71.95   same as the book
tf-idf: F1(neg)                        this run      49.41   book      49.41   same as the book
tf-idf: dims                           this run     30,195   book     30,195   same as the book
tf-idf 1-2 gram: accuracy              this run      92.57   book      92.57   same as the book
tf-idf 1-2 gram: macro-F1              this run      76.21   bo

**Reading it.** On this corpus and with these settings, TF-IDF over word unigrams and
bigrams has the highest macro-F1, and plain counts score above TF-IDF. The bigram feature
count, 101,659, is the fingerprint [chapter 12](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/12-classification.html) used to recover the exact split. The
differences between adjacent rows are one to three points; section 7 puts intervals on
them.

The word analyzer and the character analyzer see different things in the same string.
The chapter's example:

In [10]:
sample = "5/5 harika!! ürün çok i̇yi 👍"
word_analyzer = schemes["tf-idf"].build_analyzer()
char_analyzer = schemes["tf-idf char_wb 3-5"].build_analyzer()
print("### the same string through the two analyzers")
print(f'input:  "{sample}"\n')
print('analyzer="word", token_pattern=[a-zçğıöşü]+')
print(" ", word_analyzer(sample))
print('\nanalyzer="char_wb", ngram_range=(3, 5)     (first few of the 3-grams)')
three_grams = [g for g in char_analyzer(sample) if len(g) == 3]
print(" ", three_grams[:11], "...")
print()
for label, ours, book in [("word tokens", word_analyzer(sample), ['harika', 'ürün', 'çok', 'i', 'yi']),
                          ("first eleven char 3-grams", three_grams[:11],
                           [' 5/', '5/5', '/5 ', ' ha', 'har', 'ari', 'rik', 'ika', 'ka!', 'a!!', '!! '])]:
    print(f"{label}: " + ("same as the book" if ours == book else "DIFFERS from the book"))

### the same string through the two analyzers
input:  "5/5 harika!! ürün çok i̇yi 👍"

analyzer="word", token_pattern=[a-zçğıöşü]+
  ['harika', 'ürün', 'çok', 'i', 'yi']

analyzer="char_wb", ngram_range=(3, 5)     (first few of the 3-grams)
  [' 5/', '5/5', '/5 ', ' ha', 'har', 'ari', 'rik', 'ika', 'ka!', 'a!!', '!! '] ...

word tokens: same as the book
first eleven char 3-grams: same as the book


### It made no difference on this corpus

The Turkish-aware lowercasing changed almost nothing here, and the chapter measures why:
the reviews contain almost no uppercase İ, but the *file itself* already contains the
combining dot, in text that was lowercased with the wrong rule before it was published.

**What to look at:** how many reviews contain U+0307, and how many vocabulary columns
exist only because the mark splits a word in two.

In [11]:
DOT = "\u0307"
all_text = "".join(df["text"])
vs_book("characters in the de-duplicated file", len(all_text), 37_076_865, "{:,}")
vs_book("uppercase characters", sum(c.isupper() for c in all_text), 0)
del all_text

has_dot = sub["text"].str.contains(DOT)
vs_book("reviews with U+0307 in the 50,000", int(has_dot.sum()), 4_194, "{:,}")
vs_book("  share, %", 100 * has_dot.mean(), 8.4, "{:.1f}")
after_i = sum(s_[i - 1] == "i" for s_ in sub["text"] for i, c in enumerate(s_) if c == DOT and i > 0)
total_marks = sum(s_.count(DOT) for s_ in sub["text"])
print(f"U+0307 occurrences: {total_marks:,}, of which after the letter i: {after_i:,}")

vocab = set(schemes["bag of words"].vocabulary_)
repaired = CountVectorizer(min_df=2, token_pattern=PAT, **LOC).fit(train["text"].str.replace(DOT, ""))
only_original = vocab - set(repaired.vocabulary_)
only_repaired = set(repaired.vocabulary_) - vocab
print()
vs_book("vocabulary, text as distributed", len(vocab), 30_195, "{:,}")
vs_book("vocabulary, U+0307 deleted", len(repaired.vocabulary_), 29_823, "{:,}")
vs_book("columns that exist only with the mark", len(only_original), 595, "{:,}")
vs_book("forms that appear only after the repair", len(only_repaired), 223, "{:,}")
print("  fragments:", ", ".join(sorted(only_original)[:10]))
print("  restored: ", ", ".join(sorted(only_repaired)[:10]))

example = next(s_ for s_ in train["text"] if "tavsi" + DOT + "ye" in s_)
print(f"\n  {example[:60]!r}\n  -> {tokenize(example)[:10]}")

characters in the de-duplicated file   this run 37,076,865   book 37,076,865   same as the book


uppercase characters                   this run          0   book          0   same as the book
reviews with U+0307 in the 50,000      this run      4,194   book      4,194   same as the book
  share, %                             this run        8.4   book        8.4   same as the book
U+0307 occurrences: 17,736, of which after the letter i: 17,736



vocabulary, text as distributed        this run     30,195   book     30,195   same as the book
vocabulary, U+0307 deleted             this run     29,823   book     29,823   same as the book
columns that exist only with the mark  this run        595   book        595   same as the book
forms that appear only after the repair this run        223   book        223   same as the book
  fragments: ade, adem, ai, alabi, alinabi, alişveri, annemi, ari, arni, arçeli
  restored:  abimi, alişverişe, alişverişim, alişverişin, alişverişler, annemgile, babylis, bahsettikleri, bebeğimizde, bebişimi

  'si̇pari̇ş verdi̇ği̇m gün kargoya veri̇ldi̇. türki̇ye ni̇n bi'
  -> ['si', 'pari', 'ş', 'verdi', 'ği', 'm', 'gün', 'kargoya', 'veri', 'ldi']


**Reading it.** `tr_lower` does not delete a combining dot that is already in the text.
The repair is to delete the mark from the text you were given, not to lowercase it
differently. What the repair does to the scores is the chapter's exercise 7, and
exercise 2 at the end of this notebook.

### One change at a time

"TF-IDF" is three decisions at once: the IDF weight, the logarithm of the term
frequency, and L2 normalisation of each row. Plain counts scored higher than TF-IDF in
round 1, so the chapter changes one decision at a time, on the same unigram vocabulary.

**What to look at:** the drop comes with L2 normalisation, not with IDF. The classifier's
regularisation strength `C` is fixed at 1.0 in every row, and normalised rows have much
smaller values.

In [12]:
C_train = X_train["bag of words"].astype(np.float64)
C_test = X_test["bag of words"].astype(np.float64)

def log_tf(X):
    X = X.copy()
    X.data = 1 + np.log(X.data)
    return X

idf = TfidfTransformer(norm=None, use_idf=True, sublinear_tf=False).fit(C_train)
idf_log = TfidfTransformer(norm=None, use_idf=True, sublinear_tf=True).fit(C_train)
ablation = [
    ('raw counts  (= "bag of words")', "no ", "no ", "none", C_train, C_test, (92.14, 73.51, 51.30)),
    ("counts + L2", "no ", "no ", " L2 ", normalize(C_train), normalize(C_test), (88.07, 69.57, 45.85)),
    ("log-TF + L2", "no ", "yes", " L2 ", normalize(log_tf(C_train)), normalize(log_tf(C_test)), (88.37, 70.00, 46.53)),
    ("counts + idf + L2", "yes", "no ", " L2 ", normalize(idf.transform(C_train)), normalize(idf.transform(C_test)), (90.20, 72.16, 49.74)),
    ("log-TF + idf, no norm", "yes", "yes", "none", idf_log.transform(C_train), idf_log.transform(C_test), (92.67, 71.83, 47.61)),
    ('log-TF + idf + L2  (= "tf-idf")', "yes", "yes", " L2 ", normalize(idf_log.transform(C_train)), normalize(idf_log.transform(C_test)), (90.07, 71.95, 49.41)),
]
print("### one change at a time, unigram word features, min_df=2")
print(f"{'':<34}idf   log-TF   norm   accuracy  macro-F1  F1(neg)   book macro-F1")
for name, use_idf, use_log, norm, A, B, book in ablation:
    acc, macro, f1n = scores(y_test, new_clf().fit(A, y_train).predict(B))
    same = "same" if f"{macro:.2f}" == f"{book[1]:.2f}" and f"{acc:.2f}" == f"{book[0]:.2f}" else "DIFFERS"
    print(f"{name:<34}{use_idf}    {use_log}     {norm}  {acc:8.2f}  {macro:8.2f}  {f1n:7.2f}   {book[1]:8.2f}  {same}")

### one change at a time, unigram word features, min_df=2
                                  idf   log-TF   norm   accuracy  macro-F1  F1(neg)   book macro-F1


raw counts  (= "bag of words")    no     no      none     92.14     73.51    51.30      73.51  same
counts + L2                       no     no       L2      88.07     69.57    45.85      69.57  same


log-TF + L2                       no     yes      L2      88.37     70.00    46.53      70.00  same
counts + idf + L2                 yes    no       L2      90.20     72.16    49.74      72.16  same
log-TF + idf, no norm             yes    yes     none     92.67     71.83    47.61      71.83  same
log-TF + idf + L2  (= "tf-idf")   yes    yes      L2      90.07     71.95    49.41      71.95  same


So the fair comparison gives each representation its own `C`. The chapter chooses it on a
validation split carved out of the training set (32,000 to fit, 8,000 to validate), then
refits on all 40,000 with the chosen value and scores the test set once.

**What to look at:** the order of the two rows reverses. With `C` chosen per
representation, TF-IDF scores above raw counts.

In [13]:
C_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]
fit_part, val_part = train_test_split(train, test_size=8_000, stratify=train["label"], random_state=42)
tuned_pred = {}
print("### C chosen on a validation split carved out of train (32,000 fit / 8,000 val)")
for name, book_C, book_f1 in [("bag of words", 1.0, 73.51), ("tf-idf", 10.0, 74.71)]:
    vec = make_schemes()[name]
    X_fit, X_val = vec.fit_transform(fit_part["text"]), vec.transform(val_part["text"])
    val_f1 = {C: f1_score(val_part["label"], new_clf(C).fit(X_fit, fit_part["label"]).predict(X_val),
                          average="macro") for C in C_GRID}
    best_C = max(val_f1, key=val_f1.get)
    print(f"{name}: validation macro-F1 by C  " + "  ".join(f"{C:g}:{100 * v:.2f}" for C, v in val_f1.items()))
    tuned_pred[name] = new_clf(best_C).fit(X_train[name], y_train).predict(X_test[name])
    vs_book(f"  {name}: best C on validation", best_C, book_C)
    vs_book(f"  {name}: test macro-F1", scores(y_test, tuned_pred[name])[1], book_f1, "{:.2f}")

### C chosen on a validation split carved out of train (32,000 fit / 8,000 val)


bag of words: validation macro-F1 by C  0.01:62.88  0.03:66.27  0.1:68.98  0.3:70.63  1:71.63  3:70.84  10:70.11  30:69.54  100:69.74


  bag of words: best C on validation   this run        1.0   book        1.0   same as the book
  bag of words: test macro-F1          this run      73.51   book      73.51   same as the book


tf-idf: validation macro-F1 by C  0.01:58.10  0.03:60.75  0.1:64.59  0.3:67.89  1:70.71  3:71.58  10:72.22  30:71.61  100:71.10
  tf-idf: best C on validation         this run       10.0   book       10.0   same as the book
  tf-idf: test macro-F1                this run      74.71   book      74.71   same as the book


Is that 1.2-point difference more than the test set's sampling noise? The two models
were scored on the same 10,000 reviews, so the comparison is *paired*: resample the test
reviews with replacement, score both models on each resample, and look at the
distribution of the difference. `paired_bootstrap` is used for every interval in this
notebook. It works on the four counts of each model's confusion matrix, so 10,000
resamples take seconds. The resampled positions stay in memory for the rest of the
notebook (about 400 MB).

McNemar's exact test looks only at the reviews on which the two models disagree about
being right.

In [14]:
# N_BOOT resamples of the 10,000 test positions, drawn as the chapter drew them; kept as int32
BOOT_IDX = np.random.default_rng(SEED).integers(0, len(y_test), (N_BOOT, len(y_test))).astype(np.int32)


def macro_f1_boot(y_true, y_pred, idx=BOOT_IDX, chunk=500):
    """Macro-F1 (percent) of y_pred on every bootstrap resample, from confusion counts."""
    cell = (2 * y_true + y_pred).astype(np.int8)       # 0: both negative, 1: (neg, pos), 2: (pos, neg), 3: both positive
    out = np.empty(len(idx))
    for start in range(0, len(idx), chunk):
        c = cell[idx[start:start + chunk]]
        tn, fp, fn, tp = [(c == k).sum(axis=1) for k in range(4)]
        with np.errstate(divide="ignore", invalid="ignore"):
            f1_pos = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), 0.0)
            f1_neg = np.where(2 * tn + fn + fp > 0, 2 * tn / (2 * tn + fn + fp), 0.0)
        out[start:start + chunk] = 100 * (f1_pos + f1_neg) / 2
    return out


def paired_bootstrap(y_true, pred_a, pred_b):
    """Observed macro-F1 difference a - b and its 95% bootstrap interval."""
    d = macro_f1_boot(y_true, pred_a) - macro_f1_boot(y_true, pred_b)
    observed = scores(y_true, pred_a)[1] - scores(y_true, pred_b)[1]
    return observed, np.percentile(d, 2.5), np.percentile(d, 97.5)


a, b = tuned_pred["tf-idf"], tuned_pred["bag of words"]
only_a = int(((a == y_test) & (b != y_test)).sum())
only_b = int(((a != y_test) & (b == y_test)).sum())
observed, low, high = paired_bootstrap(y_test, a, b)
note = "fewer resamples" if SKIP_HEAVY else None
print(f"### tf-idf (C chosen) against raw counts (C chosen), same 10,000 test reviews ({N_BOOT:,} resamples)")
vs_book("only tf-idf right", only_a, 151)
vs_book("only raw counts right", only_b, 80)
print(f"exact McNemar test on the two counts: p = {binomtest(only_a, only_a + only_b, 0.5).pvalue:.1e}   (book: p < 0.001)")
vs_book("macro-F1 difference", observed, 1.20, "{:.2f}")
vs_book("95% interval, lower end", low, 0.42, "{:.2f}", note=note)
vs_book("95% interval, upper end", high, 1.99, "{:.2f}", note=note)

### tf-idf (C chosen) against raw counts (C chosen), same 10,000 test reviews (10,000 resamples)
only tf-idf right                      this run        151   book        151   same as the book
only raw counts right                  this run         80   book         80   same as the book
exact McNemar test on the two counts: p = 3.5e-06   (book: p < 0.001)
macro-F1 difference                    this run       1.20   book       1.20   same as the book
95% interval, lower end                this run       0.42   book       0.42   same as the book
95% interval, upper end                this run       1.99   book       1.99   same as the book


### What the model learned, in words

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#what-the-model-learned-in-words)

A linear model over word features can be read. The chapter prints the largest
coefficients of the unigram TF-IDF model (`C = 1.0`, the round 1 row).

**What to look at:** the negative side is full of negated verbs (`etmiyorum`, `etmem`,
`beğenmedim`, `kalmadım`). Turkish puts negation inside the word, so a word-level model
gets a separate feature for each negated form it has seen.

In [15]:
unigram_model = new_clf().fit(X_train["tf-idf"], y_train)
terms = schemes["tf-idf"].get_feature_names_out()
weights = unigram_model.coef_[0]                  # classes_ are [0, 1]: positive weight -> label 1
order = np.argsort(weights)

BOOK_POS = [("gayet", 8.04), ("ederim", 6.94), ("mükemmel", 6.13), ("teşekkürler", 6.09), ("harika", 5.93),
            ("memnunum", 5.84), ("güzel", 5.48), ("süper", 4.98), ("kaliteli", 4.32)]
BOOK_NEG = [("etmiyorum", -8.16), ("değil", -6.95), ("kötü", -6.63), ("etmem", -6.35), ("beğenmedim", -6.21),
            ("iade", -5.85), ("fakat", -5.52), ("kalitesiz", -5.47), ("kalmadım", -5.33)]
print("### tf-idf + logistic regression, largest coefficients")
print(f"{'towards POSITIVE':<24}{'towards NEGATIVE'}")
for i in range(12):
    p, n = order[-1 - i], order[i]
    print(f"{terms[p]:<12} {weights[p]:+6.2f}     {terms[n]:<12} {weights[n]:+6.2f}")
print()
lookup = dict(zip(terms, weights))
same = sum(f"{lookup[w]:+.2f}" == f"{c:+.2f}" for w, c in BOOK_POS + BOOK_NEG)
print(f"the chapter prints 18 of these coefficients; {same} of 18 are the same at two decimals")

### tf-idf + logistic regression, largest coefficients
towards POSITIVE        towards NEGATIVE
gayet         +8.04     etmiyorum     -8.16
ederim        +6.94     değil         -6.95
mükemmel      +6.13     kötü          -6.63
teşekkürler   +6.09     etmem         -6.35
harika        +5.93     beğenmedim    -6.21
memnunum      +5.84     iade          -5.85
gerçekten     +5.79     fakat         -5.52
güzel         +5.48     kalitesiz     -5.47
süper         +4.98     kalmadım      -5.33
ediyorum      +4.86     zayıf         -4.78
rahat         +4.52     almayın       -4.62
kaliteli      +4.32     ama           -4.28

the chapter prints 18 of these coefficients; 18 of 18 are the same at two decimals


## 6. Round 2: the chapter 3 methods

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#round-2-the-chapter-3-methods)

Three embedding models, each trained on the 40,000 training reviews only, with the
chapter's settings. A review becomes one vector of 100 numbers:

- **averaged word2vec**: the mean of the vectors of the review's words that are in the
  vocabulary; a review with none gets the zero vector;
- **averaged fastText**: the mean over *all* the review's words, because fastText builds
  a vector for an unseen word from its character n-grams;
- **doc2vec (PV-DBOW)**: a vector learned per training review, and inferred for each test
  review.

The same logistic regression then reads the vectors.

**Reproducibility.** The chapter prints the table of its first run and says a second run
of the same code with the same seed gave 61.37, 61.29 and 55.44 macro-F1. The cell
compares with both. In the environment this notebook was saved in, word2vec and fastText
reproduce the chapter's *second* run exactly, and doc2vec changes every time the notebook
is run. The reason for doc2vec is in `gensim` 4.4.0 itself: `infer_vector` starts each
test document from a vector seeded with Python's built-in `hash` of the document's words,
and Python randomises string hashes per process unless `PYTHONHASHSEED` is set. So the
doc2vec row is one draw. Training takes about a minute.

`quiet_gensim` collects a note that `gensim` 4.4.0 sometimes prints while training
(`Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'`). No Python
exception occurred and training is not affected; chapter 3's notebook explains the cause.

In [16]:
import contextlib
import io

from gensim.models import FastText, Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument


@contextlib.contextmanager
def quiet_gensim(what):
    """Collect gensim's 'Exception ignored in our_dot_float' notes and print a count instead."""
    buffer = io.StringIO()
    with contextlib.redirect_stderr(buffer):
        yield
    lines = buffer.getvalue().splitlines()
    notes = sum("Exception ignored in" in line for line in lines)
    if notes:
        print(f"({what}: gensim printed its 'Exception ignored in our_dot_float' note {notes} times)")
    for line in lines:
        if line.strip() and "Exception ignored in" not in line:
            print(line)


def average(kv, docs, use_oov):
    """One row per document: the mean of its word vectors (zero vector if it has none)."""
    X = np.zeros((len(docs), kv.vector_size), dtype=np.float32)
    for i, doc in enumerate(docs):
        words = doc if use_oov else [w for w in doc if w in kv.key_to_index]
        if words:
            X[i] = np.mean([kv[w] for w in words], axis=0)
    return X


timing = {}
with quiet_gensim("round 2"):
    start = time.perf_counter()
    # the chapter's three models
    w2v = Word2Vec(train_tok, vector_size=100, window=5, min_count=2,
                   sg=1, negative=5, epochs=10, workers=1, seed=42)
    timing["averaged word2vec"] = time.perf_counter() - start
    start = time.perf_counter()
    ft  = FastText(train_tok, vector_size=100, window=5, min_count=2,
                   sg=1, negative=5, epochs=10, workers=1, seed=42,
                   min_n=3, max_n=6, bucket=500_000)
    timing["averaged fastText"] = time.perf_counter() - start
    start = time.perf_counter()
    d2v = Doc2Vec([TaggedDocument(d, [i]) for i, d in enumerate(train_tok)],
                  vector_size=100, min_count=2, epochs=20, dm=0,
                  negative=5, workers=1, seed=42)
    timing["doc2vec (PV-DBOW)"] = time.perf_counter() - start

    dense_test_seconds = {}
    start = time.perf_counter()
    X_train["averaged word2vec"] = average(w2v.wv, train_tok, use_oov=False)
    t0 = time.perf_counter(); X_test["averaged word2vec"] = average(w2v.wv, test_tok, use_oov=False)
    dense_test_seconds["averaged word2vec"] = time.perf_counter() - t0
    X_train["averaged fastText"] = average(ft.wv, train_tok, use_oov=True)
    t0 = time.perf_counter(); X_test["averaged fastText"] = average(ft.wv, test_tok, use_oov=True)
    dense_test_seconds["averaged fastText"] = time.perf_counter() - t0
    X_train["doc2vec (PV-DBOW)"] = np.vstack([d2v.dv[i] for i in range(len(train_tok))])
    t0 = time.perf_counter(); X_test["doc2vec (PV-DBOW)"] = np.vstack([d2v.infer_vector(d) for d in test_tok])
    dense_test_seconds["doc2vec (PV-DBOW)"] = time.perf_counter() - t0

DENSE = ["averaged word2vec", "averaged fastText", "doc2vec (PV-DBOW)"]
BOOK_ROUND2 = {"averaged word2vec": (79.92, 61.39, 34.64), "averaged fastText": (79.64, 61.36, 34.79),
               "doc2vec (PV-DBOW)": (71.96, 55.13, 27.66)}
BOOK_RERUN = {"averaged word2vec": 61.37, "averaged fastText": 61.29, "doc2vec (PV-DBOW)": 55.44}
dense_clf = {}
print("### round 2: coordinates")
print(f"{'':<22}accuracy   macro-F1   F1(neg)   dims   embed train")
for name in DENSE:
    dense_clf[name] = new_clf().fit(X_train[name], y_train)
    pred[name] = dense_clf[name].predict(X_test[name])
    acc, macro, f1n = scores(y_test, pred[name])
    print(f"{name:<22}{acc:8.2f} {macro:10.2f} {f1n:9.2f} {X_train[name].shape[1]:>6} {timing[name]:9.1f} s")
print(f"\nword2vec vocabulary {len(w2v.wv):,} words")
for name in DENSE:
    acc, macro, f1n = scores(y_test, pred[name])
    tolerance = 1.0 if name.startswith("doc2vec") else 0.5      # doc2vec inference is hash-seeded
    vs_book_near(f"{name}: accuracy", acc, BOOK_ROUND2[name][0], tolerance)
    vs_book_near(f"{name}: macro-F1", macro, BOOK_ROUND2[name][1], tolerance)
    vs_book_near(f"  macro-F1 against the chapter's re-run", macro, BOOK_RERUN[name], tolerance)

(round 2: gensim printed its 'Exception ignored in our_dot_float' note 15 times)
### round 2: coordinates
                      accuracy   macro-F1   F1(neg)   dims   embed train
averaged word2vec        79.86      61.37     34.65    100      14.6 s
averaged fastText        79.56      61.29     34.70    100      32.4 s
doc2vec (PV-DBOW)        72.45      55.58     28.20    100       9.2 s

word2vec vocabulary 30,738 words
averaged word2vec: accuracy            this run      79.86   book      79.92   -0.06 from the book, inside ±0.5
averaged word2vec: macro-F1            this run      61.37   book      61.39   -0.02 from the book, inside ±0.5
  macro-F1 against the chapter's re-run this run      61.37   book      61.37   +0.00 from the book, inside ±0.5
averaged fastText: accuracy            this run      79.56   book      79.64   -0.08 from the book, inside ±0.5
averaged fastText: macro-F1            this run      61.29   book      61.36   -0.07 from the book, inside ±0.5
  macro-F1 ag

**Reading it.** On this corpus and with these settings, all three embedding rows are
below all four count rows on macro-F1 and above the majority baseline. word2vec and
fastText are within a tenth of a point of each other.

How much of a dense row is the training seed? The chapter changed only the embedding seed
(five seeds) and reports the range. The next cell repeats that.

**Heavy cell**: ten trainings, about four minutes, more than half of this notebook's
runtime. Under `ANLP_SKIP_HEAVY=1` it trains one seed.

In [17]:
SEEDS = [1] if SKIP_HEAVY else [1, 2, 3, 4, 5]
spread = {"averaged word2vec": [], "averaged fastText": []}
start = time.perf_counter()
with quiet_gensim("seed spread"):
    for seed in SEEDS:
        m = Word2Vec(train_tok, vector_size=100, window=5, min_count=2,
                     sg=1, negative=5, epochs=10, workers=1, seed=seed)
        p = new_clf().fit(average(m.wv, train_tok, False), y_train).predict(average(m.wv, test_tok, False))
        spread["averaged word2vec"].append(scores(y_test, p)[1])
        m = FastText(train_tok, vector_size=100, window=5, min_count=2,
                     sg=1, negative=5, epochs=10, workers=1, seed=seed,
                     min_n=3, max_n=6, bucket=500_000)
        p = new_clf().fit(average(m.wv, train_tok, True), y_train).predict(average(m.wv, test_tok, True))
        spread["averaged fastText"].append(scores(y_test, p)[1])
print(f"### macro-F1 over embedding seeds {SEEDS}   ({time.perf_counter() - start:.0f}s)")
BOOK_SPREAD = {"averaged word2vec": (61.57, 62.06), "averaged fastText": (60.86, 61.68)}
for name, values in spread.items():
    print(f"{name:<20} " + "  ".join(f"{v:.2f}" for v in values)
          + f"   range {min(values):.2f}–{max(values):.2f}   book {BOOK_SPREAD[name][0]:.2f}–{BOOK_SPREAD[name][1]:.2f}"
          + ("   [one seed only: ANLP_SKIP_HEAVY=1]" if SKIP_HEAVY else ""))

(seed spread: gensim printed its 'Exception ignored in our_dot_float' note 73 times)
### macro-F1 over embedding seeds [1, 2, 3, 4, 5]   (247s)
averaged word2vec    61.58  61.97  62.06  61.70  61.57   range 61.57–62.06   book 61.57–62.06
averaged fastText    61.51  61.21  60.86  61.68  61.31   range 60.86–61.68   book 60.86–61.68


Both rounds in one table, sorted by macro-F1, as the chapter prints it.

In [18]:
ALL = ["bag of words", "tf-idf", "tf-idf 1-2 gram", "tf-idf char_wb 3-5"] + DENSE
print("### both rounds, sorted by macro-F1")
for name in sorted(ALL, key=lambda n: -scores(y_test, pred[n])[1]):
    acc, macro, f1n = scores(y_test, pred[name])
    print(f"{name:<22}{acc:8.2f} {macro:10.2f} {f1n:9.2f} {X_train[name].shape[1]:>9,}")
print(f"{'majority baseline':<22}{majority_acc:8.2f} {majority_macro:10.2f} {0:9.2f}         —")

### both rounds, sorted by macro-F1
tf-idf 1-2 gram          92.57      76.21     56.47   101,659
bag of words             92.14      73.51     51.30    30,195
tf-idf                   90.07      71.95     49.41    30,195
tf-idf char_wb 3-5       88.63      70.39     47.14    94,939
averaged word2vec        79.86      61.37     34.65       100
averaged fastText        79.56      61.29     34.70       100
doc2vec (PV-DBOW)        72.45      55.58     28.20       100
majority baseline        93.68      48.37      0.00         —


## 7. Intervals: the paired bootstrap

The table above is one test set. The chapter puts a 95% interval on the macro-F1
difference of every adjacent pair of rows, with the paired bootstrap of section 5: the
same resampled test reviews for both models.

**What to look at:** which intervals contain zero. The pairs between two sparse rows use
the same resamples as the chapter (seed 42) and reproduce exactly. The chapter computed
the pairs that involve a dense row from its second run, so those are compared within a
tolerance; the pair with doc2vec moves with that row.

In [19]:
BOOK_PAIRS = [("tf-idf 1-2 gram", "bag of words", 2.69, 1.60, 3.82),
              ("bag of words", "tf-idf", 1.56, 0.53, 2.55),
              ("tf-idf", "tf-idf char_wb 3-5", 1.57, 0.59, 2.54),
              ("tf-idf char_wb 3-5", "averaged word2vec", 9.01, 7.97, 10.01),
              ("averaged word2vec", "averaged fastText", 0.08, -0.60, 0.77),
              ("averaged fastText", "doc2vec (PV-DBOW)", 5.85, 5.02, 6.69),
              ("tf-idf 1-2 gram", "averaged word2vec", 14.83, 13.65, 16.01)]
print(f"### paired bootstrap on the 10,000 test reviews: macro-F1 difference, 95% interval ({N_BOOT:,} resamples)")
for a, b, b_obs, b_low, b_high in BOOK_PAIRS:
    observed, low, high = paired_bootstrap(y_test, pred[a], pred[b])
    ours = f"{observed:+6.2f}   [{low:+6.2f}, {high:+6.2f}]"
    book = f"{b_obs:+6.2f}   [{b_low:+6.2f}, {b_high:+6.2f}]"
    if SKIP_HEAVY:
        verdict = "not comparable: fewer resamples"
    elif a in DENSE or b in DENSE:
        verdict = "dense row: " + ("inside" if max(abs(observed - b_obs), abs(low - b_low), abs(high - b_high)) <= 0.5 else "OUTSIDE") + " ±0.5 of the book"
    else:
        verdict = "same as the book" if ours == book else "DIFFERS from the book"
    print(f"{a:<19} - {b:<19} {ours}   book {book}   {verdict}")

### paired bootstrap on the 10,000 test reviews: macro-F1 difference, 95% interval (10,000 resamples)


tf-idf 1-2 gram     - bag of words         +2.69   [ +1.60,  +3.82]   book  +2.69   [ +1.60,  +3.82]   same as the book


bag of words        - tf-idf               +1.56   [ +0.53,  +2.55]   book  +1.56   [ +0.53,  +2.55]   same as the book


tf-idf              - tf-idf char_wb 3-5   +1.57   [ +0.59,  +2.54]   book  +1.57   [ +0.59,  +2.54]   same as the book


tf-idf char_wb 3-5  - averaged word2vec    +9.01   [ +7.97, +10.01]   book  +9.01   [ +7.97, +10.01]   dense row: inside ±0.5 of the book


averaged word2vec   - averaged fastText    +0.08   [ -0.60,  +0.77]   book  +0.08   [ -0.60,  +0.77]   dense row: inside ±0.5 of the book


averaged fastText   - doc2vec (PV-DBOW)    +5.71   [ +4.89,  +6.54]   book  +5.85   [ +5.02,  +6.69]   dense row: inside ±0.5 of the book


tf-idf 1-2 gram     - averaged word2vec   +14.83   [+13.65, +16.01]   book +14.83   [+13.65, +16.01]   dense row: inside ±0.5 of the book


**Reading it.** The gap between the two families, and the adjacent gaps inside the sparse
family, are larger than the sampling noise of this test set. The gap between word2vec and
fastText is not: its interval contains zero. These intervals cover the sampling of the
*test* reviews only. They say nothing about another training split, and for the dense
rows the seed spread measured above comes on top.

## 8. The out-of-vocabulary test

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#the-out-of-vocabulary-test)

word2vec has a vector only for the forms it kept, here those seen at least twice in
training. The chapter counts how much of the test set falls outside that vocabulary, by
token and by distinct form. These counts depend on the vocabulary only, so they reproduce
exactly.

**What to look at:** 7% of the tokens and 46% of the distinct forms.

In [20]:
kv, fv = w2v.wv, ft.wv
test_tokens = [w for doc in test_tok for w in doc]
test_forms = set(test_tokens)
oov_tokens = sum(w not in kv.key_to_index for w in test_tokens)
oov_forms = sorted(w for w in test_forms if w not in kv.key_to_index)
empty = [i for i, doc in enumerate(test_tok) if not any(w in kv.key_to_index for w in doc)]

print("### coverage of the 10,000 test reviews")
vs_book("test tokens", len(test_tokens), 229_055, "{:,}")
vs_book("outside the word2vec vocabulary", oov_tokens, 16_660, "{:,}")
vs_book("  share of tokens, %", 100 * oov_tokens / len(test_tokens), 7.27, "{:.2f}")
vs_book("distinct test word forms", len(test_forms), 32_573, "{:,}")
vs_book("outside the vocabulary", len(oov_forms), 15_098, "{:,}")
vs_book("  share of forms, %", 100 * len(oov_forms) / len(test_forms), 46.4, "{:.1f}")
vs_book("test reviews with no usable token", len(empty), 7)

never_tokens = sum(counts[w] == 0 for w in test_tokens)
never_forms = sum(counts[w] == 0 for w in test_forms)
print("\n### the same counts, split by whether the form was ever seen in training")
vs_book("test tokens never seen in training", never_tokens, 11_681, "{:,}")
vs_book("  share of tokens, %", 100 * never_tokens / len(test_tokens), 5.10, "{:.2f}")
vs_book("distinct forms never seen", never_forms, 10_921, "{:,}")
vs_book("  share of forms, %", 100 * never_forms / len(test_forms), 33.5, "{:.1f}")
vs_book("seen once, cut by min_count=2", len(oov_forms) - never_forms, 4_177, "{:,}")

### coverage of the 10,000 test reviews
test tokens                            this run    229,055   book    229,055   same as the book
outside the word2vec vocabulary        this run     16,660   book     16,660   same as the book
  share of tokens, %                   this run       7.27   book       7.27   same as the book
distinct test word forms               this run     32,573   book     32,573   same as the book
outside the vocabulary                 this run     15,098   book     15,098   same as the book
  share of forms, %                    this run       46.4   book       46.4   same as the book
test reviews with no usable token      this run          7   book          7   same as the book

### the same counts, split by whether the form was ever seen in training
test tokens never seen in training     this run     11,681   book     11,681   same as the book
  share of tokens, %                   this run       5.10   book       5.10   same as the book
distinct forms never s

The seven reviews with no usable token become the zero vector under word2vec averaging.
The chapter reports that all seven are labelled positive and all seven are classified
negative, because the classifier's intercept is negative.

In [21]:
zero_vector_p = dense_clf["averaged word2vec"].predict_proba(np.zeros((1, 100)))[0, 1]
print("texts:    ", [test["text"].iloc[i][:30] for i in empty])
print("gold:     ", y_test[empty].tolist())
print("predicted:", pred["averaged word2vec"][empty].tolist())
vs_book_near("intercept", float(dense_clf["averaged word2vec"].intercept_[0]), -0.83, 0.15)
print(f"P(positive | zero vector) = {zero_vector_p:.2f}")

texts:     ['yanilmişim...', 'avantajlari', 'kuşadası', 'murat', 'mukemmelllll', 'selametle ..', 'haaarika!']
gold:      [1, 1, 1, 1, 1, 1, 1]
predicted: [0, 0, 0, 0, 0, 0, 0]
intercept                              this run      -0.83   book      -0.83   +0.00 from the book, inside ±0.15
P(positive | zero vector) = 0.30


fastText builds a vector for any form from its character n-grams, so its lookup coverage
is 100%. The chapter measures how much of an unseen form's n-gram material the model has
actually seen: for each of the 15,098 out-of-vocabulary forms, the share of its
3-to-6-character n-grams that also occur in at least one vocabulary word.

In [22]:
def char_ngrams(word, low=3, high=6):
    s = f"<{word}>"
    return [s[i:i + n] for n in range(low, high + 1) for i in range(len(s) - n + 1)]


seen_ngrams = {g for w in fv.index_to_key for g in char_ngrams(w)}
share_seen = np.array([np.mean([g in seen_ngrams for g in char_ngrams(w)]) for w in oov_forms])
vs_book("mean share of n-grams seen, %", 100 * share_seen.mean(), 84.5, "{:.1f}")
vs_book("forms with every n-gram seen, %", 100 * (share_seen == 1).mean(), 24.7, "{:.1f}")
vs_book("forms with fewer than half seen, %", 100 * (share_seen < 0.5).mean(), 4.8, "{:.1f}")

mean share of n-grams seen, %          this run       84.5   book       84.5   same as the book
forms with every n-gram seen, %        this run       24.7   book       24.7   same as the book
forms with fewer than half seen, %     this run        4.8   book        4.8   same as the book


Now the neighbours. Three forms that never occur in training, and then two common words.
The lists are five or six nearest neighbours by cosine. The chapter's lists come from
its first run, whose vectors differ slightly from the ones trained above, so the cell
prints how many of the chapter's neighbours appear in this run's list and not a match.

**What to look at:** word2vec raises a `KeyError` on an unseen form. For a common word,
word2vec returns related words from other stems, and fastText returns forms that are
close in spelling.

In [23]:
def neighbours(model, word, k):
    return [w for w, _ in model.most_similar(word, topn=k)]


def vs_book_list(label, ours, book):
    shared = len(set(ours) & set(book))
    print(f"  {label:<9} {', '.join(ours)}\n  {'':<9} [{shared} of the book's {len(book)} are in this list]")


print("### nearest neighbours for unseen word forms")
BOOK_UNSEEN = {"kargolanmamış": ["kargolanmış", "kargolandı", "kargolanma", "kargolanıp", "kargolu"],
               "tavsiyelerim": ["tavsiyeleri", "tavsiyeler", "tavsiyeederim", "tavsiyedir"],
               "hızlıcana": ["hızlıca", "hızlıda", "hızlısı", "hızlıydı", "hızlı"]}
for word, book in BOOK_UNSEEN.items():
    print(f"{word}   (occurrences in training: {counts[word]})")
    try:
        kv[word]
    except KeyError:
        print("  word2vec  KeyError")
    vs_book_list("fastText", neighbours(fv, word, 5), book)

print("\n### nearest neighbours for a common word")
BOOK_COMMON = {"kargo": (["teslimat", "gönderim", "gönderi", "kargoda", "kargoyla", "kargolama"],
                         ["kargocu", "kargolu", "kargosu", "kargonuz", "kargom", "kargomun"]),
               "kalitesiz": (["dandik", "kullanışsız", "dayanıksız", "uyduruk", "kırılacak"],
                             ["kalitesizmiş", "kalitesizliği", "kalitel", "kalitesi", "kalitesindeki"])}
for word, (book_w2v, book_ft) in BOOK_COMMON.items():
    print(f"{word}   (occurrences in training: {counts[word]:,})")
    vs_book_list("word2vec", neighbours(kv, word, len(book_w2v)), book_w2v)
    vs_book_list("fastText", neighbours(fv, word, len(book_ft)), book_ft)

print(f"\n### kalitesizmiş — in the word2vec vocabulary, seen {counts['kalitesizmiş']} times in training")
print("  word2vec ", ", ".join(neighbours(kv, "kalitesizmiş", 5)))
print("  fastText ", ", ".join(neighbours(fv, "kalitesizmiş", 4)))

### nearest neighbours for unseen word forms
kargolanmamış   (occurrences in training: 0)
  word2vec  KeyError
  fastText  kargolanmış, kargolandı, kargolanma, kargolanıp, kargolu
            [5 of the book's 5 are in this list]
tavsiyelerim   (occurrences in training: 0)
  word2vec  KeyError
  fastText  tavsiyeleri, tavsiyeler, tavsiyeederim, şüpelerim, yeğlerim
            [3 of the book's 4 are in this list]
hızlıcana   (occurrences in training: 0)
  word2vec  KeyError
  fastText  hızlıca, hızlıda, hızlısı, hızlıydı, hızlı
            [5 of the book's 5 are in this list]

### nearest neighbours for a common word
kargo   (occurrences in training: 2,590)
  word2vec  teslimat, gönderim, gönderi, kargoda, kargoyla, kargolama
            [6 of the book's 6 are in this list]
  fastText  kargocu, kargolu, kargosu, kargonuz, kargom, kargomun
            [6 of the book's 6 are in this list]
kalitesiz   (occurrences in training: 152)
  word2vec  dandik, kullanışsız, dayanıksız, dikişler, uydu

**Reading it.** `kalitesizmiş` is inside the word2vec vocabulary, so the lookup raises no
error, but the form was seen twice, and its word2vec neighbours reflect neither the
sentiment nor the form of the word. The chapter's point is that past the vocabulary
boundary the failure is loud, and just inside it the failure is silent.

Two words are an illustration. The chapter measures the spelling tendency on 500 words
drawn from the vocabulary words with at least six letters and at least 20 occurrences in
training: the share of each model's top-5 neighbours that share the query's first five
(or four) characters.

In [24]:
eligible = [w for w in kv.index_to_key if len(w) >= 6 and counts[w] >= 20]
sample_words = list(np.random.default_rng(42).choice(eligible, 500, replace=False))
vs_book("eligible vocabulary words", len(eligible), 2_806, "{:,}")
BOOK_PREFIX = {(5, "word2vec"): 6.5, (5, "fastText"): 60.7, (4, "word2vec"): 8.6, (4, "fastText"): 67.4}
for k in (5, 4):
    for name, model in [("word2vec", kv), ("fastText", fv)]:
        share = np.mean([np.mean([n[:k] == w[:k] for n in neighbours(model, w, 5)]) for w in sample_words])
        vs_book_near(f"first {k} characters shared, {name}, %", 100 * share, BOOK_PREFIX[(k, name)], 3.0, "{:.1f}")

eligible vocabulary words              this run      2,806   book      2,806   same as the book


first 5 characters shared, word2vec, % this run        6.5   book        6.5   -0.02 from the book, inside ±3.0


first 5 characters shared, fastText, % this run       60.7   book       60.7   -0.02 from the book, inside ±3.0


first 4 characters shared, word2vec, % this run        8.6   book        8.6   +0.04 from the book, inside ±3.0


first 4 characters shared, fastText, % this run       67.4   book       67.4   +0.04 from the book, inside ±3.0


## 9. Error examples for understanding the score gap

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#why-the-dense-methods-lost)

The table says the count methods score higher here; it does not say why. The chapter
looks at the test reviews on which plain word TF-IDF and averaged fastText *disagree*.
With two classes, whenever they disagree exactly one of them is right, and those two
counts are what McNemar's test uses.

**What to look at:** the split of the disagreements, and then who the reviews in each
group are.

In [25]:
a, b = pred["tf-idf"], pred["averaged fastText"]
only_tfidf = (a == y_test) & (b != y_test)
only_ft = (b == y_test) & (a != y_test)
n_a, n_b = int(only_tfidf.sum()), int(only_ft.sum())
chi2 = (abs(n_a - n_b) - 1) ** 2 / (n_a + n_b)
observed, low, high = paired_bootstrap(y_test, a, b)

vs_book_near("same label, % of test reviews", 100 * (a == b).mean(), 84.1, 0.5, "{:.1f}")
vs_book_near("only tf-idf right", n_a, 1_315, 30, "{:,}")
vs_book_near("only fastText right", n_b, 272, 30, "{:,}")
vs_book_near("McNemar chi-squared (corrected)", chi2, 684.2, 40, "{:.1f}")
vs_book_near("macro-F1 gap", observed, 10.66, 0.5)
print(f"  paired bootstrap 95% [{low:.2f}, {high:.2f}]   book [9.52, 11.77]")

tfidf_vocab = schemes["tf-idf"].vocabulary_
n_tok = np.array([len(doc) for doc in test_tok])
n_oov = np.array([sum(w not in tfidf_vocab for w in doc) for doc in test_tok])
print(f"\n{'':<22}{'reviews':>8}{'gold positive':>16}{'tokens outside the tf-idf vocabulary':>40}")
for name, mask in [("all test reviews", np.ones(len(y_test), bool)), ("both right", (a == y_test) & (b == y_test)),
                   ("only tf-idf right", only_tfidf), ("only fastText right", only_ft)]:
    print(f"{name:<22}{mask.sum():>8,}{100 * y_test[mask].mean():>15.1f}%{100 * n_oov[mask].sum() / n_tok[mask].sum():>39.2f}%")

same label, % of test reviews          this run       84.0   book       84.1   -0.05 from the book, inside ±0.5
only tf-idf right                      this run      1,323   book      1,315   +8.00 from the book, inside ±30
only fastText right                    this run        272   book        272   +0.00 from the book, inside ±30
McNemar chi-squared (corrected)        this run      691.2   book      684.2   +7.02 from the book, inside ±40
macro-F1 gap                           this run      10.66   book      10.66   +0.00 from the book, inside ±0.5
  paired bootstrap 95% [9.52, 11.77]   book [9.52, 11.77]

                       reviews   gold positive    tokens outside the tf-idf vocabulary
all test reviews        10,000           93.7%                                   7.32%
both right               7,684           94.0%                                   6.51%
only tf-idf right        1,323           98.0%                                   9.48%
only fastText right        272      

**Reading it.** Out-of-vocabulary forms are more frequent among the disagreements than
among the reviews both models get right, but about equally frequent on both sides, so
they do not separate which model turns out right. The class does: the reviews only
fastText gets right are much more often negative than the test set as a whole, and the
reviews only TF-IDF gets right are almost all positive. The chapter reads that as the
averaged-embedding setup working at higher recall and lower precision on the negative
class.

The chapter prints six selected disagreements. Here are the same six texts with the two
models' predictions in *this* run (0 = negative, 1 = positive). The chapter chose them
from its first run; a row where both columns agree here was a disagreement there. Six
chosen examples show kinds of text, not rates.

In [26]:
BOOK_EXAMPLES = ["en guvenli alısveriş sitesi", "çok hizli ve temi̇z tesli̇mat", "çok sessiz çalışıyor makina .",
                 "sıkıcı edebi yonu cok zayif", "koku ne çok kötü ne çok iyi", "cok guzel! kedi bayildi buna"]
texts = test["text"].tolist()
print(f"{'text':<34}{'gold':>5}{'tf-idf':>8}{'fastText':>10}")
for example in BOOK_EXAMPLES:
    if example in texts:
        i = texts.index(example)
        print(f"{example:<34}{y_test[i]:>5}{a[i]:>8}{b[i]:>10}")
    else:
        print(f"{example:<34}  not found as a whole test review")

text                               gold  tf-idf  fastText
en guvenli alısveriş sitesi           1       0         1
çok hizli ve temi̇z tesli̇mat         1       0         1
çok sessiz çalışıyor makina .         1       0         1
sıkıcı edebi yonu cok zayif           0       0         1
koku ne çok kötü ne çok iyi           0       0         1
cok guzel! kedi bayildi buna          1       1         1


### A review and its negation

The chapter tests the representations directly: five reviews and their negations, and the
cosine between the two members of each pair.

**What to look at:** under averaged fastText a review and its negation are close to each
other. Under TF-IDF they are much further apart, because the negated verb is a
different column. The TF-IDF column reproduces exactly; the fastText column depends on
the trained vectors.

In [27]:
def doc_vector(text, model):
    return np.mean([model[w] for w in tokenize(text)], axis=0)


def cos(x, y):
    return float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))


PAIRS = [("ürünü çok beğendim", "ürünü hiç beğenmedim", 0.776, 0.193),
         ("kesinlikle tavsiye ederim", "kesinlikle tavsiye etmem", 0.919, 0.474),
         ("kargo çok hızlı geldi", "kargo çok yavaş geldi", 0.943, 0.572),
         ("fiyatına göre gayet kaliteli", "fiyatına göre hiç kaliteli değil", 0.892, 0.690),
         ("bu üründen memnun kaldım", "bu üründen memnun kalmadım", 0.962, 0.582)]
print("### cosine between a review and its negation")
print(f"{'':<62}{'fastText':>9}{'book':>7}{'tf-idf':>9}{'book':>7}")
for positive, negative, book_ft, book_tfidf in PAIRS:
    T = schemes["tf-idf"].transform([positive, negative])
    c_ft = cos(doc_vector(positive, fv), doc_vector(negative, fv))
    c_tfidf = cosine_similarity(T[0], T[1])[0, 0]
    flag = "" if f"{c_tfidf:.3f}" == f"{book_tfidf:.3f}" else "  tf-idf DIFFERS"
    print(f"{positive + ' / ' + negative:<62}{c_ft:>9.3f}{book_ft:>7.3f}{c_tfidf:>9.3f}{book_tfidf:>7.3f}{flag}")

### cosine between a review and its negation
                                                               fastText   book   tf-idf   book
ürünü çok beğendim / ürünü hiç beğenmedim                         0.776  0.776    0.193  0.193
kesinlikle tavsiye ederim / kesinlikle tavsiye etmem              0.919  0.919    0.474  0.474
kargo çok hızlı geldi / kargo çok yavaş geldi                     0.943  0.943    0.572  0.572
fiyatına göre gayet kaliteli / fiyatına göre hiç kaliteli değil    0.892  0.892    0.690  0.690
bu üründen memnun kaldım / bu üründen memnun kalmadım             0.962  0.962    0.582  0.582


## 10. The crossover: how the order depends on the number of labels

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#the-crossover-when-embeddings-do-win)

Everything so far used 40,000 labels. The chapter repeats the comparison with 100 to
40,000 labelled reviews, on the same test set.

**The protocol matters here.** The representations stay fixed: the vocabulary, the IDF
weights and the embeddings are all learned from the *text* of all 40,000 training
reviews, which needs no labels. Only the classifier sees fewer labels. That is the
situation of someone with a lot of unlabelled text and few labels. If the representation
were also learned from the labelled subset only, the curve would be a different one; the
chapter's exercise 12 asks for that, and [chapter 5's budget grid](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html) uses that other protocol.

First one stratified label sample per size (seed 42), as in the chapter's first table.

**What to look at:** below 1,000 labels the TF-IDF rows sit at the majority baseline
(48.37) and the embedding rows do not. By 2,500 labels the order has changed.

In [28]:
CURVE = ["tf-idf", "tf-idf char_wb 3-5", "averaged word2vec", "averaged fastText"]
positions = np.arange(len(y_train))
SPARSE_CSR = {name: X_train[name].tocsr() for name in ["tf-idf", "tf-idf 1-2 gram", "tf-idf char_wb 3-5"]}


def label_sample(n, seed):
    if n == len(y_train):
        return positions
    chosen, _ = train_test_split(positions, train_size=n, stratify=y_train, random_state=seed)
    return chosen


def curve_point(name, chosen):
    X = SPARSE_CSR[name] if name in SPARSE_CSR else X_train[name]
    p = new_clf().fit(X[chosen], y_train[chosen]).predict(X_test[name])
    return scores(y_test, p)[1]


BOOK_CURVE = {100: (48.37, 48.37, 53.51, 53.87), 250: (48.53, 48.84, 61.59, 63.10), 500: (49.28, 50.60, 61.02, 61.16),
              1_000: (55.40, 58.55, 59.52, 59.93), 2_500: (66.61, 68.36, 60.91, 61.19), 5_000: (71.16, 70.47, 61.52, 60.80),
              10_000: (72.55, 71.29, 61.80, 61.56), 20_000: (72.42, 71.26, 60.94, 60.96), 40_000: (71.95, 70.39, 61.39, 61.36)}
print("### macro-F1 by labelled training size (same test set of 10,000)")
print(f"{'n':>7} {'negatives':>10}   tf-idf   tf-idf char   word2vec   fastText    sparse columns   dense, max |Δ| from book")
for n, book in BOOK_CURVE.items():
    chosen = label_sample(n, 42)
    row = [curve_point(name, chosen) for name in CURVE]
    sparse_same = all(f"{row[i]:.2f}" == f"{book[i]:.2f}" for i in (0, 1))
    dense_gap = max(abs(row[i] - book[i]) for i in (2, 3))
    print(f"{n:>7,} {int((y_train[chosen] == 0).sum()):>10,} {row[0]:>8.2f} {row[1]:>13.2f} {row[2]:>10.2f} {row[3]:>10.2f}"
          f"    {'same as the book' if sparse_same else 'DIFFER':<16} {dense_gap:.2f}")

### macro-F1 by labelled training size (same test set of 10,000)
      n  negatives   tf-idf   tf-idf char   word2vec   fastText    sparse columns   dense, max |Δ| from book
    100          6    48.37         48.37      58.23      57.49    same as the book 4.72
    250         16    48.53         48.84      65.21      64.94    same as the book 3.62
    500         32    49.28         50.60      63.21      62.80    same as the book 2.19


  1,000         63    55.40         58.55      61.10      61.10    same as the book 1.58


  2,500        158    66.61         68.36      61.70      61.32    same as the book 0.79
  5,000        316    71.16         70.47      61.52      60.97    same as the book 0.17


 10,000        632    72.55         71.29      61.95      61.69    same as the book 0.15


 20,000      1,264    72.42         71.26      60.94      60.88    same as the book 0.08


 40,000      2,529    71.95         70.39      61.37      61.29    same as the book 0.07


**Reading it.** The two sparse columns reproduce exactly. The dense columns differ from
the chapter's table, which comes from its first run, and most at the smallest sizes: with
6 or 16 negative reviews in the labelled sample, a small difference in the vectors moves
the classifier a long way.

One label sample per size is one draw. The chapter then draws **ten** stratified label
samples at each size and reports the mean, the minimum and the maximum, adding word
bigram TF-IDF.

It is 400 classifier fits and takes about ten seconds, because the matrices are built
once and only the rows change. The chapter computed this table with the vectors of its
second run, so all four columns can be compared.

In [29]:
TEN = ["tf-idf", "tf-idf 1-2 gram", "averaged word2vec", "averaged fastText"]
SIZES = [100, 250, 500, 1_000, 1_250, 1_500, 1_750, 2_000, 2_500, 5_000]
N_SAMPLES = 10
BOOK_TEN = {100: (48.49, 48.45, 59.68, 58.66, 0), 250: (48.98, 48.70, 61.69, 60.96, 0), 500: (50.65, 49.17, 61.85, 61.96, 0),
            1_000: (56.57, 51.68, 62.15, 61.96, 0), 1_250: (59.59, 53.43, 62.01, 62.00, 2), 1_500: (61.87, 55.63, 62.06, 61.90, 4),
            1_750: (63.95, 57.81, 61.89, 61.64, 10), 2_000: (65.54, 59.69, 61.70, 61.44, 10),
            2_500: (67.52, 63.42, 61.68, 61.55, 10), 5_000: (70.97, 71.49, 61.48, 61.18, 10)}
start = time.perf_counter()
ten = {(name, n, seed): curve_point(name, label_sample(n, seed))
       for n in SIZES for seed in range(N_SAMPLES) for name in TEN}
print(f"### {N_SAMPLES} label samples per size: mean macro-F1 [min, max]; representations fixed   ({time.perf_counter() - start:.0f}s)")
print(f"{'n':>6}   {'tf-idf':<22}{'tf-idf 1-2 gram':<22}{'word2vec':<22}{'fastText':<22}tf-idf > fastText   book")
for n in SIZES:
    cells = []
    for name in TEN:
        v = [ten[(name, n, s)] for s in range(N_SAMPLES)]
        cells.append(f"{np.mean(v):5.2f} [{min(v):5.2f}, {max(v):5.2f}]")
    wins = sum(ten[("tf-idf", n, s)] > ten[("averaged fastText", n, s)] for s in range(N_SAMPLES))
    print(f"{n:>6,}   " + "".join(f"{c:<22}" for c in cells) + f"{wins:>5} / {N_SAMPLES}         {BOOK_TEN[n][4]:>2} / 10")
same = sum(f"{np.mean([ten[(name, n, s)] for s in range(10)]):.2f}" == f"{BOOK_TEN[n][i]:.2f}"
           for n in SIZES for i, name in enumerate(TEN))
print(f"\nmeans equal to the book's at two decimals: {same} of {len(SIZES) * len(TEN)}")
for n in SIZES:
    for i, name in enumerate(TEN):
        ours = np.mean([ten[(name, n, s)] for s in range(10)])
        if f"{ours:.2f}" != f"{BOOK_TEN[n][i]:.2f}":
            vs_book_near(f"  {n:>5,} labels, {name}", ours, BOOK_TEN[n][i], 0.5)

### 10 label samples per size: mean macro-F1 [min, max]; representations fixed   (7s)
     n   tf-idf                tf-idf 1-2 gram       word2vec              fastText              tf-idf > fastText   book
   100   48.49 [48.35, 48.85]  48.45 [48.37, 48.69]  59.68 [54.63, 61.68]  58.66 [53.80, 61.44]      0 / 10          0 / 10
   250   48.98 [48.35, 49.63]  48.70 [48.37, 49.17]  61.69 [58.58, 65.07]  60.96 [56.55, 64.83]      0 / 10          0 / 10
   500   50.65 [49.46, 51.94]  49.17 [48.51, 49.95]  61.85 [59.63, 64.18]  61.96 [60.47, 63.84]      0 / 10          0 / 10
 1,000   56.57 [55.14, 59.80]  51.68 [50.39, 53.36]  62.15 [61.61, 63.32]  61.96 [61.01, 63.15]      0 / 10          0 / 10
 1,250   59.59 [57.39, 62.41]  53.43 [51.60, 56.44]  62.01 [60.81, 62.91]  62.00 [60.83, 63.29]      2 / 10          2 / 10
 1,500   61.87 [60.50, 65.01]  55.63 [53.54, 58.41]  62.06 [60.69, 63.43]  61.90 [60.36, 63.52]      4 / 10          4 / 10
 1,750   63.95 [62.34, 65.35]  57.81 [56.37, 60.

**Reading it.** In the chapter's run, word TF-IDF was below averaged fastText in all ten
samples up to 1,000 labels, above it in all ten from 1,750 on, and split in between.
Word bigram TF-IDF, the highest row at 40,000 labels, needs more labels than unigram
TF-IDF to get there: it has three times as many columns to estimate. Read any ordering
that rests on a single label sample together with the `[min, max]` spread.

The crossover belongs to this protocol: representations learned from 40,000 unlabelled
texts, a from-scratch embedding of 100 dimensions, and this corpus.

## 11. What it costs

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#what-it-costs)

The chapter's cost table has three columns: the time to build the representation on
40,000 training reviews, the time to vectorise the 10,000 test reviews, and the memory of
the training matrix. The times below are this machine's, from the single runs above; the
chapter warns that repeated runs differ by about ten percent. The memory column does not
depend on the machine.

In [30]:
def matrix_mb(X):
    if sp.issparse(X):
        return (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6
    return X.nbytes / 1e6


BOOK_MB = {"tf-idf": 9.7, "tf-idf 1-2 gram": 15.4, "tf-idf char_wb 3-5": 144.8,
           "averaged word2vec": 16.0, "averaged fastText": 16.0, "doc2vec (PV-DBOW)": 16.0}
print("### cost, on one CPU core, 40,000 training documents (single run, not repeated)")
print(f"{'':<22}{'build the':>15}{'vectorise':>14}{'training':>12}")
print(f"{'':<22}{'representation':>15}{'10,000 test':>14}{'matrix':>12}{'book':>12}")
for name in BOOK_MB:
    if name in cost:
        build, vectorise = cost[name][0], cost[name][1]
    else:
        build, vectorise = timing[name], dense_test_seconds[name]
    mb = matrix_mb(X_train[name])
    same = "same" if f"{mb:.1f}" == f"{BOOK_MB[name]:.1f}" else "DIFFERS"
    print(f"{name:<22}{build:>13.1f} s{vectorise:>12.2f} s{mb:>9.1f} MB{BOOK_MB[name]:>9.1f} MB  {same}")

### cost, on one CPU core, 40,000 training documents (single run, not repeated)
                            build the     vectorise    training
                       representation   10,000 test      matrix        book
tf-idf                          0.5 s        0.11 s      9.7 MB      9.7 MB  same
tf-idf 1-2 gram                 0.9 s        0.17 s     15.4 MB     15.4 MB  same
tf-idf char_wb 3-5              2.8 s        0.68 s    144.8 MB    144.8 MB  same
averaged word2vec              14.6 s        0.10 s     16.0 MB     16.0 MB  same
averaged fastText              32.4 s        0.30 s     16.0 MB     16.0 MB  same
doc2vec (PV-DBOW)               9.2 s        2.01 s     16.0 MB     16.0 MB  same


### The hybrid

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#the-hybrid-and-an-honest-negative-result)

Concatenate character TF-IDF (spelling) with averaged fastText (meaning) and give both to
one classifier. The chapter first checks the scale of the two blocks, because one
L2-penalised classifier with a single `C` treats them according to their size.

In [31]:
dense_train, dense_test = X_train["averaged fastText"], X_test["averaged fastText"]
char_train, char_test = X_train["tf-idf char_wb 3-5"], X_test["tf-idf char_wb 3-5"]
row_norm = np.linalg.norm(dense_train, axis=1)
print("### block scales")
print(f"averaged fastText rows:  mean L2 norm {row_norm.mean():.2f}, mean |component| {np.abs(dense_train).mean():.2f}"
      "      (book: 2.5 and 0.2)")
print(f"char_wb TF-IDF rows:     L2 norm 1.00, non-zeros per row {char_train.nnz / char_train.shape[0]:.0f}, "
      f"mean non-zero value {char_train.data.mean():.2f}   (book: ~301 and 0.05)")

hybrid_train = sp.hstack([char_train, sp.csr_matrix(dense_train)]).tocsr()
hybrid_test = sp.hstack([char_test, sp.csr_matrix(dense_test)]).tocsr()
pred["hybrid"] = new_clf(max_iter=3000).fit(hybrid_train, y_train).predict(hybrid_test)
acc, macro, f1n = scores(y_test, pred["hybrid"])
print(f"\n### tf-idf char_wb 3-5  ⊕  averaged fastText   ({hybrid_train.shape[1]:,} dimensions)")
vs_book("dimensions", hybrid_train.shape[1], 95_039, "{:,}")
vs_book_near("accuracy", acc, 89.36, 0.5)
vs_book_near("macro-F1", macro, 71.65, 0.5)
vs_book_near("F1(neg)", f1n, 49.24, 0.7)
for a, b, book in [("hybrid", "tf-idf char_wb 3-5", "+1.28 [0.65, 1.92]"), ("tf-idf 1-2 gram", "hybrid", "+4.54 [3.42, 5.63]")]:
    observed, low, high = paired_bootstrap(y_test, pred[a], pred[b])
    print(f"{a} − {b}: {observed:+.2f} [{low:.2f}, {high:.2f}]   book {book}")

### block scales
averaged fastText rows:  mean L2 norm 2.53, mean |component| 0.20      (book: 2.5 and 0.2)
char_wb TF-IDF rows:     L2 norm 1.00, non-zeros per row 301, mean non-zero value 0.05   (book: ~301 and 0.05)



### tf-idf char_wb 3-5  ⊕  averaged fastText   (95,039 dimensions)
dimensions                             this run     95,039   book     95,039   same as the book
accuracy                               this run      89.42   book      89.36   +0.06 from the book, inside ±0.5
macro-F1                               this run      71.66   book      71.65   +0.01 from the book, inside ±0.5
F1(neg)                                this run      49.23   book      49.24   -0.01 from the book, inside ±0.7


hybrid − tf-idf char_wb 3-5: +1.28 [0.65, 1.92]   book +1.28 [0.65, 1.92]


tf-idf 1-2 gram − hybrid: +4.54 [3.42, 5.63]   book +4.54 [3.42, 5.63]


**Reading it.** The plain concatenation is ahead of both of its parts and behind word
bigram TF-IDF. Comparing a hybrid only with its own parts would have reported a success.

The chapter then adjusts the scales: L2-normalise the dense block row by row, multiply it
by a coefficient α, and choose α and `C` on the 32,000/8,000 validation split, refitting
on all 40,000. For a fair reference, character TF-IDF alone gets the same `C` grid.

**Heavy cell**: 24 fits on 95,000 columns, about a minute. Under `ANLP_SKIP_HEAVY=1` the
grid is the single point the chapter's search chose.

In [32]:
ALPHAS = [2.0] if SKIP_HEAVY else [0.25, 0.5, 1.0, 2.0, 4.0]
CS = [10.0] if SKIP_HEAVY else [1.0, 3.0, 10.0, 30.0]
fit_i, val_i = train_test_split(positions, test_size=8_000, stratify=y_train, random_state=42)
dense_unit_train, dense_unit_test = normalize(dense_train), normalize(dense_test)


def mix(S, D, alpha):
    return sp.hstack([S, sp.csr_matrix(D * alpha)]).tocsr()


start = time.perf_counter()
char_vec = make_schemes()["tf-idf char_wb 3-5"]
S_fit = char_vec.fit_transform(train["text"].iloc[fit_i])
S_val = char_vec.transform(train["text"].iloc[val_i])
grid = {}
for alpha in ALPHAS:
    for C in CS:
        model = new_clf(C, max_iter=3000).fit(mix(S_fit, dense_unit_train[fit_i], alpha), y_train[fit_i])
        grid[(alpha, C)] = 100 * f1_score(y_train[val_i], model.predict(mix(S_val, dense_unit_train[val_i], alpha)), average="macro")
best_alpha, best_C = max(grid, key=grid.get)
pred["hybrid tuned"] = new_clf(best_C, max_iter=3000).fit(mix(char_train, dense_unit_train, best_alpha), y_train) \
                                                    .predict(mix(char_test, dense_unit_test, best_alpha))
char_grid = {C: 100 * f1_score(y_train[val_i], new_clf(C, max_iter=3000).fit(S_fit, y_train[fit_i]).predict(S_val), average="macro")
             for C in ([30.0] if SKIP_HEAVY else CS)}
best_char_C = max(char_grid, key=char_grid.get)
pred["char tuned"] = new_clf(best_char_C, max_iter=3000).fit(char_train, y_train).predict(char_test)
print(f"### normalised blocks, α and C chosen on validation   ({time.perf_counter() - start:.0f}s)")
note = "grid reduced to one point" if SKIP_HEAVY else None
print("validation macro-F1:  " + "  ".join(f"α={a:g},C={c:g}: {v:.2f}" for (a, c), v in sorted(grid.items(), key=lambda kv_: -kv_[1])[:4]))
vs_book("chosen α", best_alpha, 2.0, note=note)
vs_book("chosen C", best_C, 10.0, note=note)
vs_book_near("tuned hybrid, test macro-F1", scores(y_test, pred["hybrid tuned"])[1], 74.69, 0.5, note=note)
vs_book("char TF-IDF alone, chosen C", best_char_C, 30.0, note=note)
vs_book("char TF-IDF alone, tuned, test macro-F1", scores(y_test, pred["char tuned"])[1], 73.46, "{:.2f}", note=note)
for a, b, book in [("hybrid tuned", "hybrid", "+3.02 [2.00, 4.06]"), ("hybrid tuned", "char tuned", "+1.23 [0.33, 2.15]"),
                   ("tf-idf 1-2 gram", "hybrid tuned", "+1.52 [0.20, 2.80]")]:
    observed, low, high = paired_bootstrap(y_test, pred[a], pred[b])
    print(f"{a} − {b}: {observed:+.2f} [{low:.2f}, {high:.2f}]   book {book}")

### normalised blocks, α and C chosen on validation   (45s)
validation macro-F1:  α=2,C=10: 73.80  α=4,C=10: 73.72  α=0.5,C=30: 73.69  α=1,C=30: 73.66
chosen α                               this run        2.0   book        2.0   same as the book
chosen C                               this run       10.0   book       10.0   same as the book
tuned hybrid, test macro-F1            this run      74.69   book      74.69   -0.00 from the book, inside ±0.5
char TF-IDF alone, chosen C            this run       30.0   book       30.0   same as the book
char TF-IDF alone, tuned, test macro-F1 this run      73.46   book      73.46   same as the book


hybrid tuned − hybrid: +3.02 [2.00, 4.06]   book +3.02 [2.00, 4.06]


hybrid tuned − char tuned: +1.23 [0.33, 2.15]   book +1.23 [0.33, 2.15]


tf-idf 1-2 gram − hybrid tuned: +1.52 [0.20, 2.80]   book +1.52 [0.20, 2.80]


**Reading it.** Adjusting the scales changes the hybrid's score by about three points, so
reading the plain concatenation as "the hybrid does not help" would have been wrong.
Given the same tuning budget, character TF-IDF alone also rises. In the chapter's run the
tuned hybrid stays behind *untuned* word bigram TF-IDF (76.21).

## 12. The limit the unigram bag and the average share: word order

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#the-failure-they-share)

Two reviews with the same words and opposite meanings. Addition does not depend on the
order of its terms, so unigram TF-IDF and a plain average of word vectors each produce
the same vector for both.

**What to look at:** the cosine is exactly 1 for the unigram bag and for the average, and
drops once bigrams are counted. Then look at what the bigram classifier does with the two
different vectors.

In [33]:
review_a, review_b = "kalite iyi fiyat kötü", "kalite kötü fiyat iyi"
print("### two reviews, same words, opposite meaning")
print(f'a = "{review_a}"      (quality good, price bad)')
print(f'b = "{review_b}"      (quality bad, price good)\n')
vs_book("cosine, averaged fastText", cos(doc_vector(review_a, fv), doc_vector(review_b, fv)), 1.0, "{:.4f}")
T = schemes["tf-idf"].transform([review_a, review_b])
vs_book("cosine, tf-idf", cosine_similarity(T[0], T[1])[0, 0], 1.0, "{:.4f}")
T2 = schemes["tf-idf 1-2 gram"].transform([review_a, review_b])
vs_book("cosine, tf-idf 1-2 gram", cosine_similarity(T2[0], T2[1])[0, 0], 0.3247, "{:.4f}")

bigram_model = new_clf().fit(X_train["tf-idf 1-2 gram"], y_train)
bigram_vocab = schemes["tf-idf 1-2 gram"].vocabulary_
print()
for gram in ["kalite iyi", "iyi fiyat", "fiyat kötü", "kalite kötü", "kötü fiyat", "fiyat iyi"]:
    j = bigram_vocab.get(gram)
    print(f"  {gram:<12} " + (f"in the vocabulary, coefficient {bigram_model.coef_[0, j]:+.2f}" if j is not None
                               else "not in the vocabulary (below min_df=2)"))
p_a, p_b = bigram_model.predict_proba(T2)[:, 1]
print()
vs_book("P(positive), review a", p_a, 0.26, "{:.2f}")
vs_book("P(positive), review b", p_b, 0.15, "{:.2f}")

### two reviews, same words, opposite meaning
a = "kalite iyi fiyat kötü"      (quality good, price bad)
b = "kalite kötü fiyat iyi"      (quality bad, price good)

cosine, averaged fastText              this run     1.0000   book     1.0000   same as the book
cosine, tf-idf                         this run     1.0000   book     1.0000   same as the book
cosine, tf-idf 1-2 gram                this run     0.3247   book     0.3247   same as the book

  kalite iyi   in the vocabulary, coefficient -0.62
  iyi fiyat    in the vocabulary, coefficient +0.94
  fiyat kötü   not in the vocabulary (below min_df=2)
  kalite kötü  in the vocabulary, coefficient -0.37
  kötü fiyat   not in the vocabulary (below min_df=2)
  fiyat iyi    in the vocabulary, coefficient -1.05

P(positive), review a                  this run       0.26   book       0.26   same as the book
P(positive), review b                  this run       0.15   book       0.15   same as the book


**Reading it.** Four of the six bigrams are in the vocabulary, so the two reviews get
different vectors under unigram + bigram TF-IDF. Different vectors do not mean a
different label, or a correct one: this model calls both reviews negative. A bag of
bigrams keeps local order only. Models that read the sequence are the subject of
[chapter 6](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/06-language-models.html) and [chapter 7](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/07-transformers.html).

## 13. Exercises

These follow [chapter 4's own exercises](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html#exercises), using the code above. The numbers in brackets are the
chapter's exercise numbers.

1. **Kill the imbalance instead** [2]. In `new_clf` (section 5), drop
   `class_weight="balanced"` and rerun sections 5 and 6. Report accuracy *and* macro-F1
   for every row. Which methods collapse to the majority class?
2. **Examine the combining-dot damage** [7]. Section 5 counted the 595 columns that exist
   only because of U+0307. Write a rule that repairs only the broken `i` + U+0307
   sequence, apply it to `train["text"]` and `test["text"]` alike, and rerun the round 1
   table. Report the change in vocabulary size (the chapter found 30,195 → 29,823) and in
   macro-F1. Write down beforehand which of the two you expect to move more.
3. **Is IDF really the culprit?** [10]. Add one rung to the ablation in section 5: repeat
   every row with `C` chosen on `fit_part` / `val_part`. How much of the normalisation
   effect survives?
4. **Measure the uncertainty, and extend it** [8]. `paired_bootstrap` gives the interval
   for any two entries of `pred`. Add McNemar's test for the accuracy difference of each
   pair in section 7, using the two off-diagonal counts as in section 9. Then repeat
   sections 2 to 7 with five different `random_state` values for the split and report how
   much of each gap comes from the split.
5. **Extend the repeated learning curves** [12]. In section 10, raise `N_SAMPLES` to 20
   and add `"tf-idf char_wb 3-5"` to `TEN`. Then change the protocol: refit the vectoriser
   on the text of the labelled sample only (`make_schemes()` inside `curve_point`) and
   report how the curve changes.
6. **Examine negation pairs** [4]. Extend `PAIRS` in section 9 with twenty pairs from
   your own domain. For each model record the cosine, the predicted label and
   `decision_function`. Find a pair that is separated correctly in spite of a high cosine.
7. **Weighted averaging with SIF** [3]. Replace `average` (section 6) with the weighted
   average of Arora et al.: weight $a / (a + p(w))$ with $a = 10^{-3}$, `p(w)` estimated
   from `counts`, and the common component learned from the training document vectors
   only. Compare with the plain average using `paired_bootstrap`.
8. **Choose a decision threshold** [9]. Take one model from `pred`, use
   `predict_proba` on `val_part`, and evaluate three thresholds. Decide which one you
   would defend to the person who has to read the false alarms, and report the test
   confusion matrix for that threshold once.

The next chapter keeps this split and this model and asks a different question: where
the labels came from, and what the scores mean if some of them are wrong
([chapter 5](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html)).

---

*Code in this notebook is MIT-licensed; the prose is CC BY 4.0. The Turkish product
reviews dataset is CC BY-SA 4.0 (Fatih Barmanbay, `fthbrmnby/turkish_product_reviews` on
Hugging Face).*